# DataMate: thử nghiệm inference trên Google Colab

Notebook này kiểm tra GPU, lấy code và dữ liệu, chạy dry-run, sau đó đánh giá một mẫu nhỏ trên tập train. Có thêm bước inference public-test có thể resume.

**Trước khi chạy:** đẩy branch code cần dùng lên GitHub; đặt `competition_data/` trong Google Drive; chọn GPU runtime. Notebook hiện phục vụ inference/benchmark, chưa fine-tune model.

In [ ]:
from pathlib import Path
from google.colab import drive

# Sửa các giá trị này nếu repo, branch hoặc vị trí dữ liệu của bạn khác.
REPO_URL = 'https://github.com/Long-uit/datamate.git'
REPO_BRANCH = 'long-commit'
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/DataMate/competition_data')
MODEL_ID = 'Qwen/Qwen2.5-Coder-7B-Instruct'
TRAIN_SAMPLE_SIZE = 20
PUBLIC_BATCH_SIZE = 10
PROJECT_ROOT = Path('/content/datamate')
DRIVE_RUN_DIR = Path('/content/drive/MyDrive/DataMate/colab_runs')

drive.mount('/content/drive')
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Repo:', REPO_URL, '| branch:', REPO_BRANCH)
print('Data:', DRIVE_DATA_DIR)
print('Model:', MODEL_ID)

## 1. GPU, source code và dữ liệu

Chọn **Runtime → Change runtime type → GPU**. Dữ liệu trên Drive cần chứa `train/` và/hoặc `public_test/`; notebook chép dữ liệu sang `/content` để xử lý nhanh hơn.

In [ ]:
import subprocess
import sys
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Chưa có GPU CUDA. Chọn GPU runtime rồi chạy lại.')
print('GPU:', torch.cuda.get_device_name(0))

if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', '--single-branch', '--branch', REPO_BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    print(f'Giữ source hiện có tại {PROJECT_ROOT}')

if not DRIVE_DATA_DIR.is_dir():
    raise FileNotFoundError(f'Không tìm thấy dữ liệu: {DRIVE_DATA_DIR}')
DATA_DEST = PROJECT_ROOT / 'competition_data'
if not DATA_DEST.exists():
    import shutil
    shutil.copytree(DRIVE_DATA_DIR, DATA_DEST)
else:
    print(f'Giữ dữ liệu hiện có tại {DATA_DEST}')
print('Train split:', (DATA_DEST / 'train').exists())
print('Public-test split:', (DATA_DEST / 'public_test').exists())
assert (DATA_DEST / 'train').exists() or (DATA_DEST / 'public_test').exists()

## 2. Cài dependencies và package

`requirements-colab.txt` bỏ PyTorch để tránh ghi đè bản đã đi cùng GPU runtime. Model ID được đặt qua biến môi trường để thống nhất giữa các lượt chạy.

In [ ]:
%cd /content/datamate
!python -m pip install -r colab/requirements-colab.txt
!python -m pip install -e . --no-deps
import os
os.environ['DATAMATE_MODEL_ID'] = MODEL_ID
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('DATAMATE_MODEL_ID =', os.environ['DATAMATE_MODEL_ID'])

## 3. Dry-run và inference mẫu trên tập train

Dry-run kiểm tra đường dẫn, schema và prompt mà không nạp LLM. Lượt kế tiếp chạy tối đa `TRAIN_SAMPLE_SIZE` câu có đáp án để xem PA. Giảm số câu nếu GPU thiếu VRAM hoặc thời gian runtime hạn chế.

In [ ]:
import shutil
TRAIN_DIR = PROJECT_ROOT / 'competition_data' / 'train'
if not TRAIN_DIR.is_dir():
    raise FileNotFoundError(f'Không tìm thấy train split: {TRAIN_DIR}')
subprocess.run([sys.executable, 'scripts/phase1_loop.py', '--split_dir', str(TRAIN_DIR), '--all', '--limit', '3', '--dry_run'], cwd=PROJECT_ROOT, check=True)

LOCAL_RUN_DIR = Path('/content/datamate_colab_runs')
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
report_path = LOCAL_RUN_DIR / 'train_sample_report.json'
submission_path = LOCAL_RUN_DIR / 'train_sample_submission.jsonl'
cmd = [sys.executable, 'scripts/phase1_loop.py', '--split_dir', str(TRAIN_DIR), '--all', '--limit', str(TRAIN_SAMPLE_SIZE), '--use_runner', '--save_report', str(report_path), '--save_submission', str(submission_path)]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
shutil.copy2(report_path, DRIVE_RUN_DIR / report_path.name)
shutil.copy2(submission_path, DRIVE_RUN_DIR / submission_path.name)
print('Artifacts saved in:', DRIVE_RUN_DIR)

## 4. Tuỳ chọn: public-test inference có thể resume

Mỗi lần chạy xử lý tối đa `PUBLIC_BATCH_SIZE` câu mới. Submission nằm trên Drive để `--resume` bỏ qua câu đã ghi ở lượt trước. Log được giữ cục bộ trong lúc chạy rồi nén thành một file trên Drive.

In [ ]:
PUBLIC_DIR = PROJECT_ROOT / 'competition_data' / 'public_test'
if not PUBLIC_DIR.is_dir():
    print('Bỏ qua: không có public_test split.')
else:
    public_output = DRIVE_RUN_DIR / 'public_test_submission.jsonl'
    local_logs = LOCAL_RUN_DIR / 'public_test_logs'
    cmd = [sys.executable, 'predict.py', '--data_dir', str(PUBLIC_DIR), '--model_id', MODEL_ID, '--limit', str(PUBLIC_BATCH_SIZE), '--no_fill_abstain', '--resume', '--output_file', str(public_output), '--log_dir', str(local_logs)]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
    if local_logs.exists():
        archive = shutil.make_archive(str(DRIVE_RUN_DIR / 'public_test_logs'), 'zip', root_dir=local_logs)
        print('Logs archived:', archive)
    print('Submission:', public_output)

## Lưu ý

- Model ID mặc định là Qwen2.5-Coder-7B-Instruct; giữ nhất quán model khi so sánh các lượt chạy.
- Model tải từ Hugging Face khi thử nghiệm cần được đóng gói sẵn trong môi trường Docker offline cuối cùng.
- Colab dùng để phát triển/benchmark; notebook này chưa triển khai fine-tuning.